# 가상 PK-01: Gazebo → FMS → Control Tower 검증 노트북

실제 Pinky 없이 `PK-01`을 **디지털 트윈 로봇**으로 등록하고, Gazebo가 그 로봇의 상태를 보내는 흐름을 단계별로 확인한다. 이 노트북은 Ubuntu + ROS 2 Jazzy에서 실행한다.

> 안전 경계: 실제 로봇ㆍOMXㆍ그리퍼는 이 노트북의 대상이 아니다. `control_system/`과 `pinky_pro/`도 수정하지 않는다. 공유기 연결 전 TCP 검증은 loopback 모의 수신기까지만 통과로 기록한다.

## 0. 이 노트북이 확인하는 코드 흐름

1. `db/seed_dev.sql`의 `devices`/`device_states`가 FMS에서 가상 Pinky의 **등록 정보**를 제공한다. 현재 seed ID는 `PINKY-01`이다.
2. `fms_gateway/app/main.py` → `repositories.py`가 `/api/v1/devices`로 이를 읽는다.
3. `trihouse_pinky_bringup/.../trihouse_gazebo_demo.launch.py`가 GazeboㆍNav2ㆍSafety Supervisor를 기동한다.
4. `trihouse_pinky_fleet/.../gateway_node.py`가 `/trihouse/status`를 NDJSON `robot_status`로 `control_host:control_port`에 보낸다.
5. `control_tower/gateway/operations_feed.py` → `http_server.py`가 Control Tower 전용 read model/UI를 제공한다. 현재는 메모리 feed이므로, FMS projection 구현 전에는 DB 연동 완료로 주장하지 않는다.

아래 순서를 건너뛰지 않는다: **FMS 등록 → Gateway 확인 → Gazebo 안전 확인 → TCP loopback → Control Tower 표시 → 통합 projection**.

In [ ]:
# Ubuntu 작업공간에 맞춰 한 번만 수정한다.
from pathlib import Path

TRIHOUSE_DIR = Path.home() / 'Trihouse'  # 예: Path('/home/ubuntu/Trihouse')
ROS_DOMAIN_ID = 51
ROBOT_ID = 'PK-01'        # ROS/GazeboㆍNDJSON 식별자
FMS_DEVICE_ID = 'PINKY-01'  # 현재 db/seed_dev.sql의 device_id
MAP_REVISION = 'demo-1'
MAP_PATH = '/absolute/path/to/map.yaml'  # 실제 map YAML 절대 경로
CONTROL_HOST = '127.0.0.1'  # 공유기 미연결 중에는 loopback만 사용
CONTROL_PORT = 8788
print(TRIHOUSE_DIR, ROBOT_ID, FMS_DEVICE_ID, ROS_DOMAIN_ID)

## 1. 사전 점검

아래 출력에서 `ros2`, `gz`, `docker`가 모두 보여야 한다. 하나라도 없으면 이후 단계는 **차단(blocked)** 으로 기록하고 설치/환경 문제를 먼저 해결한다. macOS에서는 여기까지의 문서ㆍ정적 검증만 하고 Gazebo 런타임은 Ubuntu에서 실행한다.

In [ ]:
!source /opt/ros/jazzy/setup.bash && export ROS_DOMAIN_ID={ROS_DOMAIN_ID} && command -v ros2 && command -v gz && docker compose version
!test -f "{MAP_PATH}" && echo 'map YAML: OK' || echo 'map YAML: BLOCKED - MAP_PATH를 수정하세요'
!cd "{TRIHOUSE_DIR}" && git rev-parse --short HEAD && git status --short

## 2. 가상 PK-01을 FMS에 등록하고 Gateway를 검증한다

현재 개발 seed의 가상 Pinky 등록 ID는 `PINKY-01`이고 Gazebo 기본 ID는 `PK-01`이다. 이 노트북에서는 둘을 억지로 자동 결합하지 않는다. 먼저 `PINKY-01` 등록을 확인하고 PK-01 Gazebo를 독립 검증한다. U3 projection 구현 시 하나의 명시적 매핑 또는 식별자 통일 결정을 추가해야 한다. 이 노트북은 DB 스키마를 임의 변경하지 않는다.

In [ ]:
# 테스트 DB는 3307/tmpfs를 사용하며 개발 volume을 건드리지 않는다.
!cd "{TRIHOUSE_DIR}" && docker compose -f compose.db_test.yaml up -d --wait mysql_test
!cd "{TRIHOUSE_DIR}" && FMS_DB_HOST=127.0.0.1 FMS_DB_PORT=3307 FMS_DB_USER=fms_gateway FMS_DB_PASSWORD=test_gateway_password FMS_DB_DATABASE=trihouse_fms PYTHONPATH= PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 fms_gateway/.venv/bin/pytest -c fms_gateway/pytest.ini fms_gateway/tests/integration/test_schema.py fms_gateway/tests/integration/test_read_api.py -v

In [ ]:
# 별도 터미널에서 실행할 Gateway 명령을 출력한다. Jupyter kernel을 점유하지 않는다.
print(f'''cd {TRIHOUSE_DIR}
set -a
source .env
set +a
fms_gateway/.venv/bin/uvicorn fms_gateway.app.main:create_app --factory --host "$FMS_API_HOST" --port "$FMS_API_PORT"''')
print('\n다른 터미널: curl -fsS http://127.0.0.1:8080/api/v1/devices | python3 -m json.tool')

In [ ]:
# Gateway 실행 뒤에만 수행한다. FMS_DEVICE_ID 항목이 보이면 가상 로봇 등록 gate 통과다.
!curl -fsS http://127.0.0.1:8080/health
!curl -fsS http://127.0.0.1:8080/ready
!curl -fsS http://127.0.0.1:8080/api/v1/devices | python3 -m json.tool

## 3. Gazebo PK-01을 기동하고 안전 경계를 확인한다

`robot_id=PK-01`은 관제/감사용 식별자이고, `ROS_DOMAIN_ID=51`은 DDS 격리값이다. 실제 TCP 서버가 아직 없거나 공유기 미연결이면 Gazebo 자체와 안전 검증을 위해 `control_host:=127.0.0.1`을 유지한다. 연결 실패 로그는 예상 가능한 상태이며, 주행 안전 검증을 통과로 바꾸지 않는다.

In [ ]:
!cd "{TRIHOUSE_DIR}" && source /opt/ros/jazzy/setup.bash && export ROS_DOMAIN_ID={ROS_DOMAIN_ID} && colcon build --packages-select trihouse_interfaces trihouse_pinky_bringup trihouse_pinky_fleet trihouse_pinky_safety trihouse_pinky_io trihouse_omx_adapter

In [ ]:
# 이 명령은 별도 Ubuntu terminal에서 실행한다. Ctrl+C로 종료한다.
print(f'''cd {TRIHOUSE_DIR}
source /opt/ros/jazzy/setup.bash
source install/setup.bash
export ROS_DOMAIN_ID={ROS_DOMAIN_ID}
ros2 launch trihouse_pinky_bringup trihouse_gazebo_demo.launch.py \
  robot_id:={ROBOT_ID} map_revision:={MAP_REVISION} map:={MAP_PATH} \
  control_host:={CONTROL_HOST} control_port:={CONTROL_PORT} omx_station_id:=OMX-01''')

In [ ]:
# Gazebo가 실행 중인 별도 terminal에서 확인한다. 기대값과 다르면 즉시 시나리오를 중단한다.
!source /opt/ros/jazzy/setup.bash && source "{TRIHOUSE_DIR}/install/setup.bash" && export ROS_DOMAIN_ID={ROS_DOMAIN_ID} && ros2 topic echo --once /trihouse/readiness
!source /opt/ros/jazzy/setup.bash && source "{TRIHOUSE_DIR}/install/setup.bash" && export ROS_DOMAIN_ID={ROS_DOMAIN_ID} && ros2 topic info /cmd_vel -v
# 기대: /cmd_vel publisher가 Safety Supervisor 하나. 2개 이상, readiness 실패, stale 센서는 FAIL.

## 4. 공유기 없이 TCP NDJSON loopback을 검증한다

이 서버는 **테스트 전용**이다. `GatewayNode`가 보내는 `hello`, `robot_status`, `task_event`, `heartbeat`를 출력할 뿐 명령을 보내지 않는다. 따라서 `execute_transport`나 실제 주행을 검증하지 않는다. 공유기 연결 전 성공은 ‘로컬 TCP codec/접속 검증 성공’으로만 기록한다.

In [ ]:
# 별도 terminal에서 실행: TCP 8788을 잠시 점유하는 read-only mock 수신기
print('''python3 - <<'PY'
import json, socketserver

class Handler(socketserver.StreamRequestHandler):
    def handle(self):
        for raw in self.rfile:
            try:
                print(json.loads(raw.decode('utf-8')))
            except (UnicodeDecodeError, json.JSONDecodeError) as error:
                print({'invalid_ndjson': str(error)})

class Server(socketserver.ThreadingTCPServer):
    allow_reuse_address = True

with Server(('127.0.0.1', 8788), Handler) as server:
    print('loopback mock listening on 127.0.0.1:8788')
    server.serve_forever()
PY''')

성공 기준은 mock terminal에서 `robot_id: PK-01`인 `hello`와 주기적 `heartbeat`, 상태 발행 시 `robot_status`가 보이는 것이다. 실패 시 먼저 `control_host`, `control_port`, Gazebo terminal의 `fleet_gateway` 로그, 그리고 8788 port 점유 프로세스를 확인한다. 이 mock은 종료한 뒤에만 실제 Control Tower TCP 수신기를 같은 port에 기동할 수 있다.

## 5. Control Tower UI를 독립적으로 확인한다

현재 `OperationsFeed`는 메모리 기반이다. 이 단계는 UI 계약과 표시만 확인한다. 다음 구현 단계(U3)에서 FMS device/job과 Gazebo 상태를 projection으로 연결해야 ‘DB 연동 UI’가 된다.

In [ ]:
# notebook kernel에서 실행하면 Control Tower UI가 localhost URL을 반환한다.
import sys
sys.path.insert(0, str(TRIHOUSE_DIR))
from control_tower.gateway.operations_feed import OperationsFeed, RobotView
from control_tower.gateway.http_server import OperationsHttpServer

feed = OperationsFeed()
feed.upsert_robot(RobotView(robot_id=ROBOT_ID, x=0.0, y=0.0, yaw=0.0, battery_percent=100.0, safety_state='READY', job_id='', stage='SIMULATION', error=''))
server = OperationsHttpServer(feed)
server.start()
print('브라우저에서 열기:', server.base_url)
# 종료 시: server.stop()

## 6. 통합 완료 판정과 다음 단계

아래 네 가지가 모두 확인되면 ‘가상 PK-01 기반 사전 통합 검증’을 통과로 기록한다.

- FMS Gateway `/api/v1/devices`에 현재 seed의 `PINKY-01`이 보인다.
- Gazebo PK-01이 domain 51에서 READY이고 `/cmd_vel` 발행자가 하나다.
- loopback mock이 PK-01 NDJSON 상태를 받는다.
- Control Tower UI가 PK-01 표시 계약을 렌더링한다.

아직 **미통과**인 것은 FMS ID `PINKY-01`과 ROS ID `PK-01`의 명시적 연결, FMS→Control Tower projection, Control Tower의 실제 TCP 수신 서버, 공유기/실장비 TCP 연결, 실제 OMX, PK-02/domain 52, RoboSapiens 읽기 전용 미러다. 다음 코드는 U3 projection부터 작성하며, 이후에만 Gazebo 상태가 DB 등록 정보와 자동으로 합쳐진다.